In [25]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

In [26]:
csv_path = "data/Australia_ODI_Cricketers_with_Role.csv"   # change to India/Australia/etc.

In [27]:
def load_and_standardize_csv(csv_path):
    df = pd.read_csv(csv_path)

    # -----------------------------------------
    # Clean column names
    # -----------------------------------------
    df.columns = df.columns.str.strip()

    # Replace weird dash symbols with NaN
    df.replace(['—', '-', 'NA', 'N/A', ''], np.nan, inplace=True)

    # -----------------------------------------
    # Rename common alternate columns to standard names
    # -----------------------------------------
    rename_map = {
        'NO': 'NO.1',
        'Runs.1': 'Runs_1',
        'BB': 'BBM',
        'Avg.1': 'Avg_2'
    }

    for old_col, new_col in rename_map.items():
        if old_col in df.columns and new_col not in df.columns:
            df.rename(columns={old_col: new_col}, inplace=True)

    # -----------------------------------------
    # Required standard columns
    # -----------------------------------------
    required_columns = [
        'Name', 'First', 'Last', 'Mat', 'Inn', 'NO.1',
        'Runs', 'HS', 'Avg', 'Balls', 'Mdn', 'Runs_1',
        'Wkt', 'BBM', 'Avg_2', 'Ca', 'St', 'Role'
    ]

    for col in required_columns:
        if col not in df.columns:
            if col in ['Name', 'HS', 'BBM', 'Role']:
                df[col] = ""
            else:
                df[col] = 0

    # -----------------------------------------
    # Numeric columns
    # -----------------------------------------
    numeric_cols = [
        'First', 'Last', 'Mat', 'Inn', 'NO.1', 'Runs',
        'Avg', 'Balls', 'Mdn', 'Runs_1', 'Wkt',
        'Avg_2', 'Ca', 'St'
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[numeric_cols] = df[numeric_cols].fillna(0)

    # -----------------------------------------
    # Text cleanup
    # -----------------------------------------
    df['Name'] = df['Name'].astype(str).str.strip()

    if 'Role' not in df.columns:
        df['Role'] = ""

    df['Role'] = df['Role'].astype(str).str.strip()

    # Standardize role names
    def clean_role(x):
        x = str(x).strip().lower()

        if x in ['batsman', 'batter']:
            return 'Batsman'
        elif x in ['bowler']:
            return 'Bowler'
        elif x in ['allrounder', 'all rounder', 'all-rounder']:
            return 'AllRounder'
        elif x in ['wicketkeeper', 'wicket keeper', 'wicket-keeper', 'wk']:
            return 'Wicketkeeper'
        else:
            return ''

    df['Role'] = df['Role'].apply(clean_role)

    # -----------------------------------------
    # Selected column handling
    # -----------------------------------------
    if 'Selected' not in df.columns:
        df['Selected'] = np.nan   # IMPORTANT: keep NaN if missing

    df['Selected'] = pd.to_numeric(df['Selected'], errors='coerce')

    return df

In [28]:
def create_features(df):
    df = df.copy()

    # Avoid division errors
    eps = 1e-6

    # Batting strike rate
    if 'Runs' in df.columns and 'Balls' in df.columns:
        df['strike_rate'] = np.where(df['Balls'] > 0, (df['Runs'] / df['Balls']) * 100, 0)
    else:
        df['strike_rate'] = 0

    # Batting consistency proxy
    if 'Avg' in df.columns:
        max_avg = df['Avg'].max() if df['Avg'].max() != 0 else 1
        df['consistency'] = df['Avg'] / max_avg
    else:
        df['consistency'] = 0

    # Bowling economy proxy
    # Using Runs_1 / wickets if proper overs/balls not available
    if 'Runs_1' in df.columns and 'Wkt' in df.columns:
        df['economy'] = np.where(df['Wkt'] > 0, df['Runs_1'] / (df['Wkt'] + eps), 999)
    else:
        df['economy'] = 999

    # Fill only numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df[numeric_cols] = df[numeric_cols].fillna(0)

    return df

In [29]:
def normalize_col(df, col):
    if col not in df.columns:
        return pd.Series([0] * len(df), index=df.index)

    vals = pd.to_numeric(df[col], errors='coerce').fillna(0)

    if len(vals) == 0:
        return pd.Series(dtype=float)

    if vals.max() == vals.min():
        return pd.Series([0.5] * len(vals), index=df.index)

    return (vals - vals.min()) / (vals.max() - vals.min())

In [30]:
def has_valid_selected_labels(role_df):
    """
    ML mode should be used only if:
    - Selected column exists
    - after dropping NaN, it contains at least two classes (0 and 1)
    """
    if 'Selected' not in role_df.columns:
        return False

    valid = role_df['Selected'].dropna()

    if len(valid) == 0:
        return False

    unique_vals = set(valid.astype(int).unique())

    return unique_vals == {0, 1} or len(unique_vals) >= 2

In [31]:
def rank_players_by_role(role_df, role_name):
    role_df = role_df.copy()

    if role_name == "Batsman":
        role_df['selection_probability'] = (
            0.40 * normalize_col(role_df, 'Runs') +
            0.30 * normalize_col(role_df, 'Avg') +
            0.15 * normalize_col(role_df, 'strike_rate') +
            0.15 * normalize_col(role_df, 'Mat')
        )

    elif role_name == "Bowler":
        role_df['selection_probability'] = (
            0.45 * normalize_col(role_df, 'Wkt') +
            0.20 * (1 - normalize_col(role_df, 'Avg_2')) +
            0.20 * (1 - normalize_col(role_df, 'economy')) +
            0.15 * normalize_col(role_df, 'Mat')
        )

    elif role_name == "AllRounder":
        role_df['selection_probability'] = (
            0.25 * normalize_col(role_df, 'Runs') +
            0.20 * normalize_col(role_df, 'Avg') +
            0.25 * normalize_col(role_df, 'Wkt') +
            0.15 * (1 - normalize_col(role_df, 'Avg_2')) +
            0.15 * normalize_col(role_df, 'Mat')
        )

    elif role_name == "Wicketkeeper":
        role_df['selection_probability'] = (
            0.30 * normalize_col(role_df, 'Runs') +
            0.20 * normalize_col(role_df, 'Avg') +
            0.25 * normalize_col(role_df, 'Ca') +
            0.15 * normalize_col(role_df, 'St') +
            0.10 * normalize_col(role_df, 'Mat')
        )

    else:
        role_df['selection_probability'] = 0

    role_df = role_df.sort_values(by='selection_probability', ascending=False)
    return role_df[['Name', 'Role', 'selection_probability']]

In [32]:
def generate_role_predictions(df, role_name, min_last_year=2020):
    role_df = df[df['Role'] == role_name].copy()

    # -----------------------------------
    # Freshness filter
    # -----------------------------------
    if 'Last' in role_df.columns:
        role_df = role_df[role_df['Last'] >= min_last_year].copy()

    print(f"\nChecking role: {role_name}")
    print(f"Players found after Last >= {min_last_year} filter: {len(role_df)}")

    if role_df.empty:
        print(f"No players found for role: {role_name}")
        return pd.DataFrame(columns=['Name', 'Role', 'selection_probability'])

    # ----------------------------
    # Decide mode
    # ----------------------------
    use_ml = has_valid_selected_labels(role_df)

    if use_ml:
        print(f"Using ML mode for {role_name}")
    else:
        print(f"Using ranking mode for {role_name}")

    # ----------------------------
    # Ranking mode
    # ----------------------------
    if not use_ml:
        return rank_players_by_role(role_df, role_name)

    # ----------------------------
    # ML mode
    # ----------------------------
    features = [
        'Mat', 'Inn', 'NO.1', 'Runs', 'Avg',
        'Balls', 'Mdn', 'Runs_1', 'Wkt', 'Avg_2',
        'Ca', 'St', 'strike_rate', 'consistency', 'economy'
    ]

    available_features = [col for col in features if col in role_df.columns]

    X = role_df[available_features].copy().fillna(0)

    y = role_df['Selected'].copy()
    valid_mask = y.notna()

    X = X[valid_mask]
    y = y[valid_mask].astype(int)

    # Safety fallback
    if len(X) == 0 or y.nunique() < 2:
        print(f"Falling back to ranking mode for {role_name}")
        return rank_players_by_role(role_df, role_name)

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
    model.fit(X_scaled, y)

    # Predict for all players of that role
    X_all = role_df[available_features].copy().fillna(0)
    X_all_scaled = scaler.transform(X_all)

    role_df['selection_probability'] = model.predict_proba(X_all_scaled)[:, 1]
    role_df = role_df.sort_values(by='selection_probability', ascending=False)

    return role_df[['Name', 'Role', 'selection_probability']]

In [33]:
def build_final_squad(best_batsmen, best_bowlers, best_allrounders, best_wicketkeepers):
    # Playing XI
    playing_xi = pd.concat([
        best_batsmen.head(4),
        best_allrounders.head(2),
        best_wicketkeepers.head(1),
        best_bowlers.head(4)
    ], ignore_index=True)

    # Reserves
    reserves = pd.concat([
        best_batsmen.iloc[4:5],
        best_wicketkeepers.iloc[1:2],
        best_allrounders.iloc[2:3],
        best_bowlers.iloc[4:5]
    ], ignore_index=True)

    final_squad = pd.concat([playing_xi, reserves], ignore_index=True)

    return final_squad[['Name', 'Role', 'selection_probability']]

In [34]:
def generate_squad_from_csv(csv_path, min_last_year=2020):
    # Load and standardize
    df = load_and_standardize_csv(csv_path)

    # Create engineered features
    df = create_features(df)

    print("\n===== ROLE DISTRIBUTION BEFORE FILTER =====")
    print(df['Role'].value_counts(dropna=False))

    if 'Last' in df.columns:
        print(f"\nApplying freshness filter: Last >= {min_last_year}")

    # Generate role-wise predictions with freshness filtering
    best_batsmen = generate_role_predictions(df, "Batsman", min_last_year)
    best_bowlers = generate_role_predictions(df, "Bowler", min_last_year)
    best_allrounders = generate_role_predictions(df, "AllRounder", min_last_year)
    best_wicketkeepers = generate_role_predictions(df, "Wicketkeeper", min_last_year)

    # Build final squad
    final_squad = build_final_squad(
        best_batsmen,
        best_bowlers,
        best_allrounders,
        best_wicketkeepers
    )

    return {
        "best_batsmen": best_batsmen,
        "best_bowlers": best_bowlers,
        "best_allrounders": best_allrounders,
        "best_wicketkeepers": best_wicketkeepers,
        "final_squad": final_squad
    }

In [35]:
results = generate_squad_from_csv(csv_path, min_last_year=2024)

best_batsmen = results["best_batsmen"]
best_bowlers = results["best_bowlers"]
best_allrounders = results["best_allrounders"]
best_wicketkeepers = results["best_wicketkeepers"]
final_squad = results["final_squad"]


===== ROLE DISTRIBUTION BEFORE FILTER =====
Role
Batsman         165
Bowler           54
AllRounder       18
Wicketkeeper     16
Name: count, dtype: int64

Applying freshness filter: Last >= 2024

Checking role: Batsman
Players found after Last >= 2024 filter: 0
No players found for role: Batsman

Checking role: Bowler
Players found after Last >= 2024 filter: 0
No players found for role: Bowler

Checking role: AllRounder
Players found after Last >= 2024 filter: 0
No players found for role: AllRounder

Checking role: Wicketkeeper
Players found after Last >= 2024 filter: 0
No players found for role: Wicketkeeper


In [13]:
print("BEST BATSMEN")
display(best_batsmen.head(10))

print("BEST BOWLERS")
display(best_bowlers.head(10))

print("BEST ALLROUNDERS")
display(best_allrounders.head(10))

print("BEST WICKETKEEPERS")
display(best_wicketkeepers.head(10))

print("FINAL 15-MEMBER SQUAD")
display(final_squad)

BEST BATSMEN


,Name,Role,selection_probability
74,Steve Smith,Batsman,0.729259
113,Michael Slater,Batsman,0.651920
162,Adam Voges,Batsman,0.624869
46,Graham Yallop,Batsman,0.572270
1,Ian Chappell,Batsman,0.567853
67,Greg Ritchie,Batsman,0.559715
197,Phillip Hughes,Batsman,0.545842
31,David Hookes,Batsman,0.521131
170,Callum Ferguson,Batsman,0.512696
59,John Dyson,Batsman,0.501371


BEST BOWLERS


,Name,Role,selection_probability
112,Glenn McGrath[114],Bowler,0.991687
139,Brett Lee,Bowler,0.970958
109,Shane Warne[114],Bowler,0.841642
184,Mitchell Starc,Bowler,0.757186
155,Mitchell Johnson,Bowler,0.757120
81,Craig McDermott,Bowler,0.706751
211,Adam Zampa,Bowler,0.674607
141,Nathan Bracken,Bowler,0.659903
125,Brad Hogg,Bowler,0.636088
126,Jason Gillespie,Bowler,0.607509


BEST ALLROUNDERS


,Name,Role,selection_probability
89,Steve Waugh,AllRounder,0.591927
147,Shane Watson,AllRounder,0.558756
229,Cameron Green,AllRounder,0.526303
138,Andrew Symonds,AllRounder,0.485341
134,Ian Harvey,AllRounder,0.482907
104,Mark Waugh,AllRounder,0.454976
127,Darren Lehmann,AllRounder,0.383673
201,James Faulkner,AllRounder,0.373306
82,Simon O'Donnell,AllRounder,0.347715
189,Mitchell Marsh,AllRounder,0.332359


BEST WICKETKEEPERS


,Name,Role,selection_probability
128,Adam Gilchrist[114],Wicketkeeper,0.666390
177,Tim Paine,Wicketkeeper,0.462793
68,Wayne B. Phillips,Wicketkeeper,0.434627
237,Josh Inglis,Wicketkeeper,0.418578
101,Ian Healy,Wicketkeeper,0.349681
143,Brad Haddin,Wicketkeeper,0.310993
222,Alex Carey,Wicketkeeper,0.269183
37,Richie Robinson,Wicketkeeper,0.228845
191,Matthew Wade,Wicketkeeper,0.223036
165,Luke Ronchi[171],Wicketkeeper,0.211675


FINAL 15-MEMBER SQUAD


,Name,Role,selection_probability
0,Steve Smith,Batsman,0.729259
1,Michael Slater,Batsman,0.651920
2,Adam Voges,Batsman,0.624869
3,Graham Yallop,Batsman,0.572270
4,Steve Waugh,AllRounder,0.591927
5,Shane Watson,AllRounder,0.558756
6,Adam Gilchrist[114],Wicketkeeper,0.666390
7,Glenn McGrath[114],Bowler,0.991687
8,Brett Lee,Bowler,0.970958
9,Shane Warne[114],Bowler,0.841642
